# Visual Search System - Requirements & Problem Framing

This notebook covers the initial design phase of building a Pinterest-like visual search system. We'll clarify requirements, define the problem statement, and frame it as an ML problem.

**Learning Objectives:**
- Understand visual search system functionality and use cases
- Learn to clarify requirements for image-based ML systems
- Frame visual similarity search as a machine learning problem

In [ ]:
# Standard imports for this notebook
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

---

## 1. Introduction to Visual Search Systems

### What is Visual Search?

Visual search is a technology that allows users to search for information using images instead of text. Unlike traditional text-based search, visual search systems analyze the visual content of images to find similar or related items.

### Pinterest-like Visual Search Functionality

Pinterest's visual search feature allows users to:
1. **Select a region of interest** - Users can crop or select a specific part of an image
2. **Find visually similar images** - The system retrieves images that look similar to the selected region
3. **Discover related content** - Users can explore visually related pins and products

This functionality is powered by deep learning models that understand visual content at a semantic level.

In [ ]:
# Visualization: Visual Search Workflow
def visualize_search_workflow():
    """Create a simple diagram showing the visual search workflow."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    steps = [
        ('1. User Image', 'User uploads or\nselects an image'),
        ('2. Crop Region', 'User selects\nregion of interest'),
        ('3. Process', 'System extracts\nvisual features'),
        ('4. Results', 'Similar images\nare retrieved')
    ]
    
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
    
    for ax, (title, desc), color in zip(axes, steps, colors):
        ax.add_patch(plt.Rectangle((0.1, 0.2), 0.8, 0.6, 
                                   facecolor=color, alpha=0.3, edgecolor=color, linewidth=2))
        ax.text(0.5, 0.7, title, ha='center', va='center', fontsize=12, fontweight='bold')
        ax.text(0.5, 0.4, desc, ha='center', va='center', fontsize=10)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
    
    plt.suptitle('Visual Search Workflow', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_search_workflow()

### Use Case: Discover Visually Similar Images

**Primary Use Case:**
Users want to discover visually similar images based on a query image or a selected region within an image.

**Example Scenarios:**
- A user sees a dress they like and wants to find similar styles
- A user photographs furniture and wants to find matching or similar items
- A designer searches for visual inspiration based on color palettes or patterns
- A user wants to find the source or higher quality version of an image

**Key Challenges:**
1. **Scale** - Searching through billions of images efficiently
2. **Visual Understanding** - Capturing what makes images "similar"
3. **Real-time Response** - Providing results in milliseconds
4. **Diverse Content** - Handling various image types and styles

---

## 2. Requirements Clarification

Before diving into the technical design, we need to clarify the requirements with stakeholders. This ensures we build the right system for the actual needs.

### Key Requirements

| Requirement | Description | Priority |
|------------|-------------|----------|
| **Ranking by Similarity** | Results must be ranked by visual similarity to the query image | High |
| **Image-only Focus** | System focuses solely on image content (no text/metadata matching) | High |
| **Cropping Support** | Users can select a specific region of an image to search | Medium |
| **No Personalization** | Results are based purely on visual similarity, not user preferences | Low |

In [ ]:
# Requirements Summary Dictionary
requirements = {
    'functional': {
        'ranking_by_similarity': {
            'description': 'Results ranked by visual similarity to query image',
            'priority': 'high',
            'metric': 'visual similarity score'
        },
        'image_only_focus': {
            'description': 'Focus on image content, not text or metadata',
            'priority': 'high',
            'metric': 'content-based retrieval accuracy'
        },
        'cropping_support': {
            'description': 'Support for selecting region of interest in query image',
            'priority': 'medium',
            'metric': 'region-based search accuracy'
        },
        'no_personalization': {
            'description': 'Pure visual similarity without user preference bias',
            'priority': 'low',
            'metric': 'consistency across users'
        }
    },
    'non_functional': {
        'latency': 'Sub-second response time for queries',
        'availability': '99.9% uptime',
        'scalability': 'Handle 100-200 billion images'
    }
}

print("=" * 60)
print("VISUAL SEARCH SYSTEM REQUIREMENTS")
print("=" * 60)

print("\n📋 Functional Requirements:")
for key, value in requirements['functional'].items():
    print(f"  • {key.replace('_', ' ').title()}")
    print(f"    Description: {value['description']}")
    print(f"    Priority: {value['priority'].upper()}")
    print()

### Scale Requirements

The system must handle a massive scale:

**Image Corpus:**
- **100-200 billion images** in the searchable index
- New images added continuously (millions per day)
- Images from diverse sources and categories

**Query Volume:**
- Millions of search queries per day
- Peak loads during high-traffic periods
- Global user base with varying access patterns

**Implications for Design:**
1. Efficient indexing and retrieval algorithms required
2. Distributed storage and computation necessary
3. Approximate nearest neighbor search for speed
4. Incremental index updates for new images

In [ ]:
# Scale Analysis
scale_params = {
    'total_images': 150_000_000_000,  # 150 billion (mid-range estimate)
    'embedding_dim': 256,  # typical embedding dimension
    'bytes_per_float': 4,  # float32
    'daily_new_images': 10_000_000,  # 10 million new images per day
    'daily_queries': 100_000_000,  # 100 million queries per day
}

# Calculate storage requirements
embedding_storage_bytes = scale_params['total_images'] * scale_params['embedding_dim'] * scale_params['bytes_per_float']
embedding_storage_tb = embedding_storage_bytes / (1024 ** 4)

print("📊 Scale Analysis")
print("=" * 50)
print(f"Total Images: {scale_params['total_images']:,}")
print(f"Embedding Dimension: {scale_params['embedding_dim']}")
print(f"\nEmbedding Storage Required: {embedding_storage_tb:,.1f} TB")
print(f"Daily New Images: {scale_params['daily_new_images']:,}")
print(f"Daily Queries: {scale_params['daily_queries']:,}")
print(f"\nQueries per Second (avg): {scale_params['daily_queries'] / 86400:,.0f}")

### Training Data: User Click Interactions

The primary source of training data comes from user interactions with the search system:

**Interaction Types:**
- **Clicks** - User clicks on a search result (positive signal)
- **Impressions** - Results shown but not clicked (weak negative signal)
- **Saves/Pins** - User saves an image (strong positive signal)
- **Time spent** - Engagement time with results

**Data Collection:**
```
Query Image → Search Results → User Interaction
     ↓              ↓                 ↓
  Image ID    [Result IDs]      [Click/No-Click]
```

**Training Pairs:**
- Positive pairs: (query image, clicked result)
- Negative pairs: (query image, shown but not clicked)

In [ ]:
# Example: Simulating interaction data structure
import pandas as pd

# Sample interaction data
sample_interactions = {
    'query_image_id': ['img_001', 'img_001', 'img_001', 'img_002', 'img_002'],
    'result_image_id': ['img_100', 'img_101', 'img_102', 'img_200', 'img_201'],
    'position': [1, 2, 3, 1, 2],
    'clicked': [True, False, True, True, False],
    'saved': [True, False, False, False, False],
    'timestamp': pd.date_range('2024-01-01', periods=5, freq='H')
}

df_interactions = pd.DataFrame(sample_interactions)
print("Sample User Interaction Data:")
print(df_interactions.to_string(index=False))

print("\n📈 Training Signal Interpretation:")
print("  • Clicked = True → Positive training pair")
print("  • Clicked = False, Position shown → Weak negative pair")
print("  • Saved = True → Strong positive signal")

---

## 3. Problem Statement

### ML Objective

**Goal:** Accurately retrieve and rank visually similar images given a query image.

More formally:
> Given a query image $I_q$, retrieve a ranked list of images $\{I_1, I_2, ..., I_k\}$ from a corpus $C$ such that the images are ordered by decreasing visual similarity to $I_q$.

### Success Criteria

| Metric | Target | Description |
|--------|--------|-------------|
| Precision@K | > 0.70 | Fraction of top-K results that are relevant |
| Recall@K | > 0.50 | Fraction of all relevant images in top-K |
| MRR | > 0.75 | Mean Reciprocal Rank of first relevant result |
| Latency | < 200ms | End-to-end query response time |

In [ ]:
# Define success metrics
success_metrics = {
    'offline_metrics': {
        'precision_at_k': {'target': 0.70, 'description': 'Precision at K retrieved results'},
        'recall_at_k': {'target': 0.50, 'description': 'Recall at K retrieved results'},
        'mrr': {'target': 0.75, 'description': 'Mean Reciprocal Rank'},
        'map': {'target': 0.65, 'description': 'Mean Average Precision'},
    },
    'online_metrics': {
        'click_through_rate': {'target': 0.15, 'description': 'CTR on search results'},
        'save_rate': {'target': 0.05, 'description': 'Rate of saving results'},
        'session_length': {'target': 5.0, 'description': 'Avg queries per session'},
    },
    'system_metrics': {
        'p99_latency_ms': {'target': 200, 'description': 'P99 query latency'},
        'throughput_qps': {'target': 10000, 'description': 'Queries per second'},
    }
}

print("🎯 Success Metrics")
print("=" * 50)
for category, metrics in success_metrics.items():
    print(f"\n{category.replace('_', ' ').title()}:")
    for name, details in metrics.items():
        print(f"  • {name}: {details['target']} ({details['description']})")

### System Input/Output Specification

**Input:**
- Query image (full image or cropped region)
- Image format: JPEG, PNG, WebP
- Optional: bounding box coordinates for region of interest

**Output:**
- Ranked list of similar image IDs
- Similarity scores for each result
- Metadata (optional): thumbnails, titles, source URLs

```
Input: {
    "image": <binary_image_data>,
    "crop_box": [x1, y1, x2, y2],  // optional
    "num_results": 20
}

Output: {
    "results": [
        {"image_id": "abc123", "similarity": 0.95, "url": "..."},
        {"image_id": "def456", "similarity": 0.91, "url": "..."},
        ...
    ],
    "query_time_ms": 45
}
```

In [ ]:
# API Interface Example
from dataclasses import dataclass
from typing import List, Optional, Tuple

@dataclass
class VisualSearchRequest:
    """Input specification for visual search."""
    image_data: bytes  # Raw image bytes
    crop_box: Optional[Tuple[int, int, int, int]] = None  # (x1, y1, x2, y2)
    num_results: int = 20
    
@dataclass
class SearchResult:
    """Individual search result."""
    image_id: str
    similarity_score: float
    thumbnail_url: Optional[str] = None
    
@dataclass
class VisualSearchResponse:
    """Output specification for visual search."""
    results: List[SearchResult]
    query_time_ms: float
    embedding_computed: bool = True

# Example usage
print("📥 Visual Search API Interface")
print("=" * 50)
print("\nRequest Schema:")
print(f"  {VisualSearchRequest.__annotations__}")
print("\nResponse Schema:")
print(f"  {VisualSearchResponse.__annotations__}")

---

## 4. ML Problem Framing

### Framing as a Ranking Problem

Visual search is fundamentally a **ranking problem**:
- Given a query, rank all candidate images by similarity
- Return the top-K most similar images

### Approach: Representation Learning

We use **representation learning** to solve this ranking problem:

1. **Learn Embeddings**: Train a neural network to map images to a dense vector space
2. **Measure Similarity**: Use distance metrics (cosine similarity, Euclidean distance) in embedding space
3. **Efficient Search**: Use approximate nearest neighbor algorithms for fast retrieval

```
Image → CNN/Transformer → Embedding Vector (256-2048 dims)
                              ↓
                    Similarity Search
                              ↓
                    Ranked Results
```

In [ ]:
# Visualization: Representation Learning Pipeline
def visualize_embedding_pipeline():
    """Visualize the image to embedding pipeline."""
    fig, ax = plt.subplots(1, 1, figsize=(14, 4))
    
    # Pipeline stages
    stages = [
        ('Image\n(224×224×3)', 0.1, '#3498db'),
        ('CNN/ViT\nBackbone', 0.3, '#2ecc71'),
        ('Feature\nExtraction', 0.5, '#e74c3c'),
        ('Embedding\n(256-d vector)', 0.7, '#9b59b6'),
        ('Index &\nSearch', 0.9, '#f39c12')
    ]
    
    for label, x, color in stages:
        ax.add_patch(plt.Rectangle((x-0.08, 0.3), 0.14, 0.4, 
                                   facecolor=color, alpha=0.3, 
                                   edgecolor=color, linewidth=2, 
                                   joinstyle='round'))
        ax.text(x, 0.5, label, ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Arrows
    for i in range(len(stages)-1):
        ax.annotate('', xy=(stages[i+1][1]-0.1, 0.5), 
                   xytext=(stages[i][1]+0.08, 0.5),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title('Visual Search: Image to Embedding Pipeline', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_embedding_pipeline()

### Embedding Space Concepts

**What is an Embedding Space?**

An embedding space is a high-dimensional vector space where:
- Each image is represented as a point (vector)
- **Similar images are close together**
- **Dissimilar images are far apart**

**Properties of a Good Embedding Space:**

| Property | Description |
|----------|-------------|
| **Semantic Similarity** | Visually similar images have similar embeddings |
| **Discrimination** | Different categories are well-separated |
| **Smoothness** | Small visual changes → small embedding changes |
| **Compactness** | Lower dimensionality while preserving information |

In [ ]:
# Demonstration: Embedding Space Visualization (2D projection)
np.random.seed(42)

# Simulate embeddings for different image categories
n_per_category = 30
categories = {
    'Dogs': (np.random.randn(n_per_category, 2) * 0.5 + np.array([2, 2]), '#e74c3c'),
    'Cats': (np.random.randn(n_per_category, 2) * 0.5 + np.array([-2, 2]), '#3498db'),
    'Cars': (np.random.randn(n_per_category, 2) * 0.5 + np.array([0, -2]), '#2ecc71'),
    'Flowers': (np.random.randn(n_per_category, 2) * 0.5 + np.array([-2, -2]), '#9b59b6'),
    'Buildings': (np.random.randn(n_per_category, 2) * 0.5 + np.array([2, -1]), '#f39c12'),
}

fig, ax = plt.subplots(figsize=(10, 8))

for category, (points, color) in categories.items():
    ax.scatter(points[:, 0], points[:, 1], c=color, label=category, alpha=0.6, s=50)

# Add a query point
query_point = np.array([1.8, 2.3])
ax.scatter(*query_point, c='black', marker='*', s=300, label='Query Image', zorder=5)

# Draw a circle for nearest neighbors
circle = plt.Circle(query_point, 1.0, fill=False, color='black', linestyle='--', linewidth=2)
ax.add_patch(circle)

ax.set_xlabel('Embedding Dimension 1', fontsize=12)
ax.set_ylabel('Embedding Dimension 2', fontsize=12)
ax.set_title('Embedding Space: Similar Images Cluster Together', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("⭐ The query image (star) finds nearest neighbors within the dashed circle")
print("   Notice how dog images cluster together, separate from other categories")

### Visual Similarity in Embedding Space

**Measuring Similarity:**

Once images are embedded, we measure similarity using distance metrics:

1. **Cosine Similarity** (most common for image embeddings):
   $$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

2. **Euclidean Distance** (L2 distance):
   $$d(\mathbf{a}, \mathbf{b}) = \|\mathbf{a} - \mathbf{b}\|_2$$

3. **Dot Product** (for normalized embeddings, equivalent to cosine):
   $$\text{sim}(\mathbf{a}, \mathbf{b}) = \mathbf{a} \cdot \mathbf{b}$$

**Why Representation Learning Works:**
- Deep neural networks learn hierarchical features
- Low-level features: edges, textures, colors
- High-level features: objects, scenes, concepts
- The final embedding captures semantic meaning

In [ ]:
# Demonstration: Similarity Metrics
from numpy.linalg import norm

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (norm(a) * norm(b))

def euclidean_distance(a, b):
    """Compute Euclidean distance between two vectors."""
    return norm(a - b)

# Example embeddings
query_embedding = np.array([0.5, 0.8, 0.2, 0.9, 0.1])
similar_embedding = np.array([0.52, 0.78, 0.22, 0.88, 0.12])  # Very similar
different_embedding = np.array([0.1, 0.2, 0.9, 0.1, 0.8])  # Very different

print("📐 Similarity Metrics Comparison")
print("=" * 50)
print(f"\nQuery embedding: {query_embedding}")
print(f"\nSimilar image embedding: {similar_embedding}")
print(f"  Cosine Similarity: {cosine_similarity(query_embedding, similar_embedding):.4f}")
print(f"  Euclidean Distance: {euclidean_distance(query_embedding, similar_embedding):.4f}")

print(f"\nDifferent image embedding: {different_embedding}")
print(f"  Cosine Similarity: {cosine_similarity(query_embedding, different_embedding):.4f}")
print(f"  Euclidean Distance: {euclidean_distance(query_embedding, different_embedding):.4f}")

print("\n✅ Higher cosine similarity = more similar images")
print("✅ Lower Euclidean distance = more similar images")

---

## Summary

In this notebook, we covered the foundational aspects of designing a visual search system:

### Key Takeaways

1. **Visual Search Use Case**: Users discover visually similar images based on a query image or selected region

2. **Requirements**:
   - Ranking by visual similarity (no personalization)
   - Image-only focus with cropping support
   - Scale: 100-200 billion images
   - Training data from user click interactions

3. **Problem Statement**:
   - ML Objective: Accurately retrieve and rank visually similar images
   - Input: Query image (with optional crop)
   - Output: Ranked list of similar images with scores

4. **ML Problem Framing**:
   - Ranking problem solved via representation learning
   - Images mapped to embedding vectors
   - Similarity measured in embedding space

### Next Steps

In the next notebook, we'll dive into:
- Data sources and schemas
- Image preprocessing operations
- Feature engineering considerations

In [ ]:
# Final Summary Visualization
print("\n" + "="*60)
print("📌 VISUAL SEARCH SYSTEM DESIGN - MODULE 2 OVERVIEW")
print("="*60)

overview = """
┌─────────────────────────────────────────────────────────┐
│  VISUAL SEARCH SYSTEM DESIGN                            │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  📷 INPUT: Query Image                                  │
│       ↓                                                 │
│  🧠 EMBEDDING: CNN/Transformer → 256-d vector           │
│       ↓                                                 │
│  🔍 SEARCH: Approximate Nearest Neighbors               │
│       ↓                                                 │
│  📊 OUTPUT: Ranked Similar Images                       │
│                                                         │
├─────────────────────────────────────────────────────────┤
│  Scale: 150B+ images | Latency: <200ms | QPS: 10K+     │
└─────────────────────────────────────────────────────────┘
"""
print(overview)